In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.linear_model import LogisticRegression

In [2]:
# --- DATABASE CONFIGURATION --- 
DB_USER = 'root'
DB_PASSWORD = 'dhyanirohit%401234'
DB_HOST = 'localhost'
DB_NAME = 'credit_risk_db'

engine = create_engine(f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")
print("Fetching Data From My SQL..")
query = """
SELECT 
    c.customer_id, c.full_name, c.annual_income, 
    l.loan_id, l.loan_amount, 
    l.credit_utilization, l.dti_ratio, l.loan_default
FROM customers c 
JOIN loans_and_credit l on c.customer_id = l.customer_id
"""
df = pd.read_sql(query, con=engine)

# Features for the model 
features = ['annual_income', 'loan_amount', 'dti_ratio', 'credit_utilization']
X=df[features]
y=df['loan_default']



Fetching Data From My SQL..


In [6]:
# --- 3. TRAIN MODEL (Simulating Production Load) --- 
print("Initilizing Model...")
model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X,y) 


Initilizing Model...


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [8]:
# --- 4. THE SCORING ENGINE (Phase 3 Core) --- 
print("Calculating Probabilty of default and Credit Scores...") 
# Get probability of class 1 (Default) 
df['probability_of_default']= model.predict_proba(X)[:, 1]

# Convert PD to a Credit Score (300 to 850)
df['credit_score'] = 850 - (df['probability_of_default'] * 550)
df['credit_score'] = df['credit_score'].astype(int)

print(df['credit_score'])
print("Done")

Calculating Probabilty of default and Credit Scores...
0       305
1       836
2       830
3       839
4       842
       ... 
4995    822
4996    836
4997    809
4998    764
4999    519
Name: credit_score, Length: 5000, dtype: int64
Done


In [12]:
# --- 5. BUSINESS DECISION RULES --- 
def assign_risk_tier(score): 
    if score >= 750:
        return 'Auto-Approve'
    elif score >= 600:
        return 'Manual Review' 
    else: 
        return 'Auto-Reject'

df['decision_tier'] = df['credit_score'].apply(assign_risk_tier)

print(df.head())

   customer_id           full_name  annual_income  loan_id  loan_amount  \
0        10000         Fitan Wable         198621    50000      1508421   
1        10001          Ethan Babu        3200578    50001      2131176   
2        10002        Aayush Sethi        2513881    50002      1839924   
3        10003  Isaac Ramakrishnan        3392191    50003      1632775   
4        10004     Vamakshi Tailor        3150819    50004      1891116   

   credit_utilization  dti_ratio  loan_default  probability_of_default  \
0                0.84       1.56             1                0.989617   
1                0.75       0.15             0                0.024149   
2                0.14       0.09             0                0.034920   
3                0.60       0.12             0                0.019479   
4                0.20       0.06             0                0.013951   

   credit_score decision_tier  
0           305   Auto-Reject  
1           836  Auto-Approve  
2       

In [14]:
# --- 6. PUSH TO DATABASE FOR POWER BI --- 
print("Pushing Final Risk Mart to MySQL for Pahse 4... ")
# Dropping loan_default as this is now a predictive dataset for the dashboard 
final_dashboard_df = df[['customer_id', 'full_name', 'loan_id', 'loan_amount', 'probability_of_default', 'credit_score', 'decision_tier']]
final_dashboard_df.to_sql('portfolio_risk_mart', con=engine, if_exists='replace', index=False)
print("✅ SUCCESS: 'portfolio_risk_mart' table created in MySQL.") 

# Let's check the distribution
print("\n--- BUSINESS OUTCOME ---") 
print(df['decision_tier'].value_counts())

Pushing Final Risk Mart to MySQL for Pahse 4... 
✅ SUCCESS: 'portfolio_risk_mart' table created in MySQL.

--- BUSINESS OUTCOME ---
decision_tier
Auto-Approve     2383
Auto-Reject      1597
Manual Review    1020
Name: count, dtype: int64
